# 01. Descripción y calidad del dataset

Este notebook caracteriza el dataset **GHAW-H: GitHub Agentic Workflow Histories**. La unidad de observación principal es una versión observada de un archivo Markdown; por eso se separan las filas versionadas de los archivos lógicos y de los repositorios.

El análisis conserva los Parquet originales y genera tablas derivadas pequeñas en la carpeta eda/data/processed/ para el segundo notebook.

## 1. Origen y carga de los datos

Fuente: [pavtch/GHAW-H en Hugging Face](https://huggingface.co/datasets/pavtch/GHAW-H). GHAW-H es una colección versionada de repositorios tempranos que adoptaron GitHub Agentic Workflows. La documentación de la fuente indica 262 repositorios, 604 historiales de archivos y 2.820 versiones Markdown observadas.

Las cinco tablas públicas se cargan desde eda/data/raw/. Cada fila representa:

- repository: un repositorio observado y sus metadatos.
- source_markdown_file_history: un archivo lógico identificado por su historial de versiones.
- source_markdown_file_snapshot: un estado inmutable del Markdown, con frontmatter y body separados.
- source_markdown_file_version: una versión ordenada que relaciona un historial con un snapshot y un commit.
- lock_file_snapshot: el archivo .lock.yml asociado a cada versión Markdown.

In [1]:
import re
import warnings
from collections.abc import Mapping

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)


from pathlib import Path


def locate_eda_dir():
    cwd = Path.cwd()
    candidates = [cwd / "eda", cwd, cwd.parent / "eda"]
    for candidate in candidates:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(
        "No se encontró eda/data/raw. Descarga las tablas siguiendo eda/README.md."
    )


EDA_DIR = locate_eda_dir()
RAW_DIR = EDA_DIR / "data" / "raw"
PROCESSED_DIR = EDA_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_URL = "https://huggingface.co/datasets/pavtch/GHAW-H"

TABLE_FILES = {
    "repository": "repository.parquet",
    "source_markdown_file_history": "source_markdown_file_history.parquet",
    "source_markdown_file_snapshot": "source_markdown_file_snapshot.parquet",
    "source_markdown_file_version": "source_markdown_file_version.parquet",
    "lock_file_snapshot": "lock_file_snapshot.parquet",
}
missing = [name for name in TABLE_FILES.values() if not (RAW_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Faltan tablas en {RAW_DIR}: {missing}")

tables = {name: pd.read_parquet(RAW_DIR / filename) for name, filename in TABLE_FILES.items()}
repositories = tables["repository"]
histories = tables["source_markdown_file_history"]
snapshots = tables["source_markdown_file_snapshot"].copy()
versions = tables["source_markdown_file_version"]
locks = tables["lock_file_snapshot"]

print(f"Fuente: {SOURCE_URL}")
print(f"Directorio de entrada: {RAW_DIR}")
print("Tablas cargadas:")
display(pd.DataFrame({"tabla": list(tables), "archivo": list(TABLE_FILES.values())}))


Fuente: https://huggingface.co/datasets/pavtch/GHAW-H
Directorio de entrada: /home/bastian/DevProjects/GitHub/tarea-miner-datos/eda/data/raw
Tablas cargadas:


,tabla,archivo
0,repository,repository.parquet
1,source_markdown_file_history,source_markdown_file_history.parquet
2,source_markdown_file_snapshot,source_markdown_file_snapshot.parquet
3,source_markdown_file_version,source_markdown_file_version.parquet
4,lock_file_snapshot,lock_file_snapshot.parquet


## 2. Descripción de las tablas y sus relaciones

El modelo es relacional y temporal. repository_id identifica un repositorio; source_markdown_file_snapshot_id identifica un contenido Markdown observado; source_markdown_file_version_id conecta ese contenido con un historial, un commit y su orden; y lock_file_snapshot conserva el artefacto compilado asociado.

En consecuencia, 2.820 filas de snapshots/versiones no equivalen a 2.820 archivos Markdown distintos: un mismo par (repository_id, path) puede aparecer varias veces.

In [2]:
def table_dimensions(table_name, frame):
    return {
        "tabla": table_name,
        "filas": len(frame),
        "columnas": len(frame.columns),
        "columnas_lista": ", ".join(frame.columns),
    }


dimensions = pd.DataFrame(
    [table_dimensions(name, frame) for name, frame in tables.items()]
)
display(dimensions)

for name, frame in tables.items():
    print(f"\n{name}")
    display(
        frame.dtypes.rename("tipo_pandas")
        .rename_axis("columna")
        .reset_index()
    )


,tabla,filas,columnas,columnas_lista
0,repository,262,12,"repository_id, url, license, repo_full_name, repo_owner, repo_name, main_lan..."
1,source_markdown_file_history,604,2,"source_markdown_file_history_id, version_count"
2,source_markdown_file_snapshot,2820,6,"source_markdown_file_snapshot_id, repository_id, path, content, frontmatter,..."
3,source_markdown_file_version,2820,8,"source_markdown_file_version_id, source_markdown_file_history_id, source_mar..."
4,lock_file_snapshot,2820,5,"lock_file_snapshot_id, source_markdown_file_version_id, path, content, file_sha"



repository


,columna,tipo_pandas
0,repository_id,int64
1,url,object
2,license,object
3,repo_full_name,object
4,repo_owner,object
5,repo_name,object
6,main_language,object
7,forks,int64
8,stars,int64
9,created_at,object



source_markdown_file_history


,columna,tipo_pandas
0,source_markdown_file_history_id,int64
1,version_count,int64


...

,columna,tipo_pandas
0,source_markdown_file_snapshot_id,int64
1,repository_id,int64
2,path,object
3,content,object
4,frontmatter,object
5,body,object



source_markdown_file_version


,columna,tipo_pandas
0,source_markdown_file_version_id,int64
1,source_markdown_file_history_id,int64
2,source_markdown_file_snapshot_id,int64
3,commit_sha,object
4,committed_at,object
5,rank,int64
6,predecessor_source_markdown_file_version_id,float64
7,successor_source_markdown_file_version_id,float64



lock_file_snapshot


,columna,tipo_pandas
0,lock_file_snapshot_id,int64
1,source_markdown_file_version_id,int64
2,path,object
3,content,object
4,file_sha,object


In [3]:
primary_keys = {
    "repository": "repository_id",
    "source_markdown_file_history": "source_markdown_file_history_id",
    "source_markdown_file_snapshot": "source_markdown_file_snapshot_id",
    "source_markdown_file_version": "source_markdown_file_version_id",
    "lock_file_snapshot": "lock_file_snapshot_id",
}

relationship_rows = [
    ("source_markdown_file_snapshot.repository_id", "repository.repository_id", "muchos a uno"),
    ("source_markdown_file_version.source_markdown_file_history_id", "source_markdown_file_history.source_markdown_file_history_id", "muchos a uno"),
    ("source_markdown_file_version.source_markdown_file_snapshot_id", "source_markdown_file_snapshot.source_markdown_file_snapshot_id", "uno a uno en esta versión"),
    ("lock_file_snapshot.source_markdown_file_version_id", "source_markdown_file_version.source_markdown_file_version_id", "uno a uno en esta versión"),
    ("source_markdown_file_version.predecessor_source_markdown_file_version_id", "source_markdown_file_version.source_markdown_file_version_id", "autorreferencia opcional"),
    ("source_markdown_file_version.successor_source_markdown_file_version_id", "source_markdown_file_version.source_markdown_file_version_id", "autorreferencia opcional"),
]
relationships = pd.DataFrame(relationship_rows, columns=["clave_foranea", "clave_referenciada", "cardinalidad"])
display(relationships)

unique_counts = pd.DataFrame(
    [
        {"entidad": "repositorios", "filas_de_la_tabla": len(repositories), "unicos": repositories.repository_id.nunique()},
        {"entidad": "versiones Markdown", "filas_de_la_tabla": len(versions), "unicos": versions.source_markdown_file_version_id.nunique()},
        {"entidad": "snapshots Markdown", "filas_de_la_tabla": len(snapshots), "unicos": snapshots.source_markdown_file_snapshot_id.nunique()},
        {"entidad": "archivos lógicos (repository_id, path)", "filas_de_la_tabla": len(snapshots), "unicos": snapshots[["repository_id", "path"]].drop_duplicates().shape[0]},
        {"entidad": "rutas sin distinguir repositorio", "filas_de_la_tabla": len(snapshots), "unicos": snapshots.path.nunique()},
    ]
)
display(unique_counts)


,clave_foranea,clave_referenciada,cardinalidad
0,source_markdown_file_snapshot.repository_id,repository.repository_id,muchos a uno
1,source_markdown_file_version.source_markdown_file_history_id,source_markdown_file_history.source_markdown_file_history_id,muchos a uno
2,source_markdown_file_version.source_markdown_file_snapshot_id,source_markdown_file_snapshot.source_markdown_file_snapshot_id,uno a uno en esta versión
3,lock_file_snapshot.source_markdown_file_version_id,source_markdown_file_version.source_markdown_file_version_id,uno a uno en esta versión
4,source_markdown_file_version.predecessor_source_markdown_file_version_id,source_markdown_file_version.source_markdown_file_version_id,autorreferencia opcional
5,source_markdown_file_version.successor_source_markdown_file_version_id,source_markdown_file_version.source_markdown_file_version_id,autorreferencia opcional


,entidad,filas_de_la_tabla,unicos
0,repositorios,262,262
1,versiones Markdown,2820,2820
2,snapshots Markdown,2820,2820
3,"archivos lógicos (repository_id, path)",2820,604
4,rutas sin distinguir repositorio,2820,359


La diferencia entre 2.820 snapshots y 604 pares (repository_id, path) es esperable: el dataset contiene historia. Para los análisis de distribución de archivos se usará el archivo lógico, mientras que para estudiar el contenido y su evolución se conservará la unidad snapshot.

## 3. Revisión de calidad

Se revisan valores ausentes, duplicados, claves primarias, claves foráneas, tipos y la calidad del YAML. Una ausencia no se marca automáticamente como error: licencias y lenguaje principal pueden no estar disponibles, y predecessor/successor son nulos en los extremos de cada historial.

In [4]:
null_rows = []
optional_nulls = {
    ("repository", "license"),
    ("repository", "main_language"),
    ("source_markdown_file_version", "predecessor_source_markdown_file_version_id"),
    ("source_markdown_file_version", "successor_source_markdown_file_version_id"),
}
for table_name, frame in tables.items():
    for column in frame.columns:
        count = int(frame[column].isna().sum())
        if count:
            null_rows.append(
                {
                    "tabla": table_name,
                    "columna": column,
                    "nulos": count,
                    "porcentaje_filas": round(count / len(frame) * 100, 2),
                    "lectura": "ausencia esperada/opcional" if (table_name, column) in optional_nulls else "revisar",
                }
            )
null_report = pd.DataFrame(null_rows)
display(null_report if not null_report.empty else pd.DataFrame([{"resultado": "No se encontraron valores ausentes"}]))


,tabla,columna,nulos,porcentaje_filas,lectura
0,repository,license,21,8.02,ausencia esperada/opcional
1,repository,main_language,1,0.38,ausencia esperada/opcional
2,source_markdown_file_version,predecessor_source_markdown_file_version_id,604,21.42,ausencia esperada/opcional
3,source_markdown_file_version,successor_source_markdown_file_version_id,604,21.42,ausencia esperada/opcional


In [5]:
pk_rows = []
for table_name, pk in primary_keys.items():
    frame = tables[table_name]
    values = frame[pk]
    if pd.api.types.is_string_dtype(values):
        empty = int(values.isna().sum() + values.fillna("").astype(str).str.strip().eq("").sum())
    else:
        empty = int(values.isna().sum())
    pk_rows.append(
        {
            "tabla": table_name,
            "clave_primaria": pk,
            "filas": len(frame),
            "nulos_o_vacios": empty,
            "repeticiones_de_clave": int(frame.duplicated(subset=[pk]).sum()),
        }
    )
pk_report = pd.DataFrame(pk_rows)
display(pk_report)

exact_duplicates = pd.DataFrame(
    [
        {"tabla": name, "filas_duplicadas_exactas": int(frame.duplicated().sum())}
        for name, frame in tables.items()
    ]
)
display(exact_duplicates)


,tabla,clave_primaria,filas,nulos_o_vacios,repeticiones_de_clave
0,repository,repository_id,262,0,0
1,source_markdown_file_history,source_markdown_file_history_id,604,0,0
2,source_markdown_file_snapshot,source_markdown_file_snapshot_id,2820,0,0
3,source_markdown_file_version,source_markdown_file_version_id,2820,0,0
4,lock_file_snapshot,lock_file_snapshot_id,2820,0,0


,tabla,filas_duplicadas_exactas
0,repository,0
1,source_markdown_file_history,0
2,source_markdown_file_snapshot,0
3,source_markdown_file_version,0
4,lock_file_snapshot,0


In [6]:
fk_specs = [
    ("source_markdown_file_snapshot", "repository_id", "repository", "repository_id"),
    ("source_markdown_file_version", "source_markdown_file_history_id", "source_markdown_file_history", "source_markdown_file_history_id"),
    ("source_markdown_file_version", "source_markdown_file_snapshot_id", "source_markdown_file_snapshot", "source_markdown_file_snapshot_id"),
    ("source_markdown_file_version", "predecessor_source_markdown_file_version_id", "source_markdown_file_version", "source_markdown_file_version_id"),
    ("source_markdown_file_version", "successor_source_markdown_file_version_id", "source_markdown_file_version", "source_markdown_file_version_id"),
    ("lock_file_snapshot", "source_markdown_file_version_id", "source_markdown_file_version", "source_markdown_file_version_id"),
]

fk_rows = []
for child_table, child_col, parent_table, parent_col in fk_specs:
    child_values = tables[child_table][child_col].dropna()
    parent_values = tables[parent_table][parent_col]
    unmatched = int((~child_values.isin(parent_values)).sum())
    fk_rows.append(
        {
            "tabla_hija": child_table,
            "columna_hija": child_col,
            "tabla_padre": parent_table,
            "columna_padre": parent_col,
            "filas_con_fk_no_nula": len(child_values),
            "sin_correspondencia": unmatched,
        }
    )
fk_report = pd.DataFrame(fk_rows)
display(fk_report)


,tabla_hija,columna_hija,tabla_padre,columna_padre,filas_con_fk_no_nula,sin_correspondencia
0,source_markdown_file_snapshot,repository_id,repository,repository_id,2820,0
1,source_markdown_file_version,source_markdown_file_history_id,source_markdown_file_history,source_markdown_file_history_id,2820,0
2,source_markdown_file_version,source_markdown_file_snapshot_id,source_markdown_file_snapshot,source_markdown_file_snapshot_id,2820,0
3,source_markdown_file_version,predecessor_source_markdown_file_version_id,source_markdown_file_version,source_markdown_file_version_id,2216,0
4,source_markdown_file_version,successor_source_markdown_file_version_id,source_markdown_file_version,source_markdown_file_version_id,2216,0
5,lock_file_snapshot,source_markdown_file_version_id,source_markdown_file_version,source_markdown_file_version_id,2820,0


In [7]:
date_checks = []
for table_name, column in [
    ("repository", "created_at"),
    ("repository", "updated_at"),
    ("repository", "pushed_at"),
    ("source_markdown_file_version", "committed_at"),
]:
    parsed = pd.to_datetime(tables[table_name][column], errors="coerce", utc=True)
    date_checks.append(
        {
            "tabla": table_name,
            "columna": column,
            "tipo_pandas": str(tables[table_name][column].dtype),
            "fechas_no_parseables": int(parsed.isna().sum()),
        }
    )

sha_checks = []
for table_name, column in [
    ("source_markdown_file_version", "commit_sha"),
    ("lock_file_snapshot", "file_sha"),
]:
    values = tables[table_name][column].dropna().astype(str)
    invalid = int((~values.str.fullmatch(r"[0-9a-fA-F]{40}")).sum())
    sha_checks.append(
        {
            "tabla": table_name,
            "columna": column,
            "tipo_pandas": str(tables[table_name][column].dtype),
            "tipo_esperado": "SHA hexadecimal de 40 caracteres",
            "inconsistencias": invalid,
        }
    )
type_checks = pd.DataFrame(date_checks + sha_checks)
display(type_checks)


,tabla,columna,tipo_pandas,fechas_no_parseables,tipo_esperado,inconsistencias
0,repository,created_at,object,0.0,NaN,NaN
1,repository,updated_at,object,0.0,NaN,NaN
2,repository,pushed_at,object,0.0,NaN,NaN
3,source_markdown_file_version,committed_at,object,0.0,NaN,NaN
4,source_markdown_file_version,commit_sha,object,NaN,SHA hexadecimal de 40 caracteres,0.0
5,lock_file_snapshot,file_sha,object,NaN,SHA hexadecimal de 40 caracteres,0.0


En los Parquet, predecessor_source_markdown_file_version_id y successor_source_markdown_file_version_id aparecen como float64 al leerlos con pandas porque contienen nulos. Se tratan como identificadores enteros anulables: la comprobación semántica verifica que todos los valores no nulos sean enteros y referencias válidas.

In [8]:
nullable_id_checks = []
for column in [
    "predecessor_source_markdown_file_version_id",
    "successor_source_markdown_file_version_id",
]:
    values = versions[column].dropna()
    non_integer = int((values % 1 != 0).sum())
    nullable_id_checks.append(
        {
            "columna": column,
            "tipo_pandas": str(versions[column].dtype),
            "no_enteros_entre_no_nulos": non_integer,
            "nulos_de_borde": int(versions[column].isna().sum()),
        }
    )
display(pd.DataFrame(nullable_id_checks))


,columna,tipo_pandas,no_enteros_entre_no_nulos,nulos_de_borde
0,predecessor_source_markdown_file_version_id,float64,0,604
1,successor_source_markdown_file_version_id,float64,0,604


In [9]:
def parse_frontmatter(raw):
    if raw is None or not str(raw).strip():
        return {}, "empty", ""
    try:
        parsed = yaml.load(str(raw), Loader=yaml.BaseLoader)
    except yaml.YAMLError as exc:
        return {}, "parse_error", str(exc).splitlines()[0]
    if parsed is None:
        return {}, "empty", ""
    if not isinstance(parsed, Mapping):
        return {}, "not_mapping", f"tipo obtenido: {type(parsed).__name__}"
    return dict(parsed), "ok", ""

parsed_results = snapshots["frontmatter"].map(parse_frontmatter)
snapshots["frontmatter_dict"] = parsed_results.map(lambda item: item[0])
snapshots["frontmatter_status"] = parsed_results.map(lambda item: item[1])
snapshots["frontmatter_error"] = parsed_results.map(lambda item: item[2])

frontmatter_quality = (
    snapshots["frontmatter_status"]
    .value_counts()
    .rename_axis("estado")
    .reset_index(name="archivos")
)
frontmatter_quality["porcentaje"] = (frontmatter_quality["archivos"] / len(snapshots) * 100).round(2)
display(frontmatter_quality)

display(
    snapshots.loc[snapshots["frontmatter_status"] != "ok", [
        "source_markdown_file_snapshot_id", "repository_id", "path", "frontmatter_status", "frontmatter_error"
    ]]
)


,estado,archivos,porcentaje
0,ok,2813,99.75
1,empty,6,0.21
2,parse_error,1,0.04


,source_markdown_file_snapshot_id,repository_id,path,frontmatter_status,frontmatter_error
991,992,4215554248678626280,.github/workflows/repo-assist.md,parse_error,while scanning a simple key
1360,1361,8454812289721828648,.github/workflows/daily-test-improver.md,empty,
1361,1362,8454812289721828648,.github/workflows/daily-test-improver.md,empty,
1362,1363,8454812289721828648,.github/workflows/daily-test-improver.md,empty,
1363,1364,8454812289721828648,.github/workflows/daily-test-improver.md,empty,
1364,1365,8454812289721828648,.github/workflows/daily-test-improver.md,empty,
1365,1366,8454812289721828648,.github/workflows/daily-test-improver.md,empty,


El frontmatter se considera un atributo del archivo que puede faltar o ser inválido. Los seis casos vacíos no se convierten en errores de datos por sí solos; el único YAML que no pudo analizarse se conserva y se marca como parse_error. Los campos opcionales ausentes se contabilizarán como ausencia, no como exclusión.

## 4. Tratamiento de los problemas encontrados

No se modifican las cinco tablas originales ni se eliminan filas. Se crea una tabla derivada por snapshot con métricas reproducibles para el EDA: longitud del body, estado del frontmatter, cantidad de campos, motor declarado y presencia de atributos. Además, se guardan dos tablas auxiliares para consultar campos y valores múltiples sin volver a depender de variables en memoria.

In [10]:
WORD_PATTERN = re.compile(r"\b[\w'-]+\b", flags=re.UNICODE)


def word_count(text):
    return len(WORD_PATTERN.findall("" if text is None else str(text)))


def attribute_values(frontmatter, field):
    if field not in frontmatter:
        return ["NO_DECLARADO"]
    value = frontmatter[field]
    if field == "engine":
        if isinstance(value, Mapping):
            return [str(value.get("id", "MAPA_SIN_ID"))]
        return [str(value)]
    if isinstance(value, Mapping):
        return [str(key) for key in value] or ["MAPA_VACIO"]
    if isinstance(value, list):
        return [str(item) for item in value] or ["LISTA_VACIA"]
    return [str(value)]


analysis = snapshots[[
    "source_markdown_file_snapshot_id", "repository_id", "path", "body", "frontmatter",
    "frontmatter_status", "frontmatter_dict",
]].copy()
analysis["frontmatter_valid"] = analysis["frontmatter_status"].eq("ok")
analysis["frontmatter_field_count"] = analysis["frontmatter_dict"].map(len)
analysis["body_word_count"] = analysis["body"].map(word_count)
analysis["body_empty"] = analysis["body"].fillna("").str.strip().eq("")
analysis["engine"] = analysis["frontmatter_dict"].map(lambda value: attribute_values(value, "engine")[0])
for field in ["on", "permissions", "tools", "safe-outputs"]:
    analysis[f"has_{field.replace('-', '_')}"] = analysis["frontmatter_dict"].map(lambda value: field in value)
analysis["file_key"] = analysis["repository_id"].astype(str) + "::" + analysis["path"]

analysis = analysis.merge(
    repositories[["repository_id", "repo_full_name", "repo_name", "main_language", "stars", "forks"]],
    on="repository_id",
    how="left",
    validate="many_to_one",
)

selected_attributes = ["on", "engine", "permissions", "tools"]
attribute_rows = []
for snapshot_id, frontmatter in zip(analysis["source_markdown_file_snapshot_id"], analysis["frontmatter_dict"]):
    for attribute in selected_attributes:
        for value in attribute_values(frontmatter, attribute):
            attribute_rows.append({"source_markdown_file_snapshot_id": snapshot_id, "attribute": attribute, "value": value})
attribute_values_table = pd.DataFrame(attribute_rows)

field_rows = []
for snapshot_id, frontmatter in zip(analysis["source_markdown_file_snapshot_id"], analysis["frontmatter_dict"]):
    for field in frontmatter:
        field_rows.append({"source_markdown_file_snapshot_id": snapshot_id, "field": str(field)})
field_presence = pd.DataFrame(field_rows, columns=["source_markdown_file_snapshot_id", "field"])

processed_columns = [
    "source_markdown_file_snapshot_id", "repository_id", "file_key", "path", "repo_full_name", "repo_name",
    "main_language", "stars", "forks", "frontmatter_status", "frontmatter_valid", "frontmatter_field_count",
    "body_word_count", "body_empty", "engine", "has_on", "has_permissions", "has_tools", "has_safe_outputs",
]
analysis_processed = analysis[processed_columns].copy()
analysis_processed.to_parquet(PROCESSED_DIR / "markdown_analysis.parquet", index=False, compression="zstd")
field_presence.to_parquet(PROCESSED_DIR / "frontmatter_fields.parquet", index=False, compression="zstd")
attribute_values_table.to_parquet(PROCESSED_DIR / "frontmatter_attribute_values.parquet", index=False, compression="zstd")

transformation_report = pd.DataFrame([
    {
        "operacion": "enriquecimiento de snapshots",
        "registros_entrada": len(snapshots),
        "registros_modificados_en_derivada": len(analysis_processed),
        "registros_excluidos": 0,
        "salida": "markdown_analysis.parquet",
    },
    {
        "operacion": "frontmatter por campo",
        "registros_entrada": len(snapshots),
        "registros_modificados_en_derivada": len(field_presence),
        "registros_excluidos": 0,
        "salida": "frontmatter_fields.parquet",
    },
    {
        "operacion": "valores de atributos seleccionados",
        "registros_entrada": len(snapshots),
        "registros_modificados_en_derivada": len(attribute_values_table),
        "registros_excluidos": 0,
        "salida": "frontmatter_attribute_values.parquet",
    },
])
display(transformation_report)
display(analysis_processed.head())
print(f"Tablas preparadas guardadas en: {PROCESSED_DIR}")


,operacion,registros_entrada,registros_modificados_en_derivada,registros_excluidos,salida
0,enriquecimiento de snapshots,2820,2820,0,markdown_analysis.parquet
1,frontmatter por campo,2820,25728,0,frontmatter_fields.parquet
2,valores de atributos seleccionados,2820,23186,0,frontmatter_attribute_values.parquet


,source_markdown_file_snapshot_id,repository_id,file_key,path,repo_full_name,repo_name,main_language,stars,forks,frontmatter_status,frontmatter_valid,frontmatter_field_count,body_word_count,body_empty,engine,has_on,has_permissions,has_tools,has_safe_outputs
0,1,9089213639968762207,9089213639968762207::.github/workflows/daily-issue-triage.md,.github/workflows/daily-issue-triage.md,apache/cloudstack,cloudstack,Java,2955,1334,ok,True,9,970,False,NO_DECLARADO,True,True,True,True
1,2,9089213639968762207,9089213639968762207::.github/workflows/daily-repo-status.md,.github/workflows/daily-repo-status.md,apache/cloudstack,cloudstack,Java,2955,1334,ok,True,7,92,False,NO_DECLARADO,True,True,True,True
2,3,9089213639968762207,9089213639968762207::.github/workflows/daily-repo-status.md,.github/workflows/daily-repo-status.md,apache/cloudstack,cloudstack,Java,2955,1334,ok,True,7,92,False,NO_DECLARADO,True,True,True,True
3,4,9089213639968762207,9089213639968762207::.github/workflows/daily-repo-status.md,.github/workflows/daily-repo-status.md,apache/cloudstack,cloudstack,Java,2955,1334,ok,True,7,91,False,NO_DECLARADO,True,True,True,True
4,5,9089213639968762207,9089213639968762207::.github/workflows/daily-repo-status.md,.github/workflows/daily-repo-status.md,apache/cloudstack,cloudstack,Java,2955,1334,ok,True,8,91,False,copilot,True,True,True,True


Tablas preparadas guardadas en: /home/bastian/DevProjects/GitHub/tarea-miner-datos/eda/data/processed


Decisión: se preservan los 2.820 snapshots, incluidos los seis frontmatters vacíos y el registro con YAML inválido. La tabla preparada no reemplaza el contenido original: solo agrega variables derivadas y deja explícito el estado de calidad. Por tanto, se modificaron/enriquecieron 2.820 registros en la salida derivada y se excluyeron 0 registros.

### Resumen de la calidad

Las comprobaciones anteriores permiten continuar al segundo notebook sin imputar licencias, idiomas ni campos de frontmatter. Las relaciones tienen que verificarse por claves, y el hecho de que un campo no esté declarado se mantiene como señal analítica.